In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Base Learners
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

# Meta Learner
from sklearn.linear_model import Ridge

# Stacking
from sklearn.ensemble import StackingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
# Load Dataset

df = pd.read_csv("../data/housing.csv")

df.head()

In [ ]:
# Dataset Info

print("Shape :", df.shape)
print("\nMissing Values :\n", df.isnull().sum())
print("\nPrice Stats :\n", df['price'].describe())

In [ ]:
# Label Encoding for Categorical Columns

encoders = {}

for col in ["location", "condition"]:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    encoders[col] = le

df.head()

In [ ]:
# Features & Target

X = df.drop("price", axis=1)

y = df["price"]

print("Features :", list(X.columns))

In [ ]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)

In [ ]:
# Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled  = scaler.transform(X_test)

print("Scaling Done ✅")

In [ ]:
# ------------------------------------------------
# Define Base Learners (Level 0 Models)
# ------------------------------------------------

base_learners = [
    ("decision_tree",  DecisionTreeRegressor(max_depth=6, random_state=42)),
    ("random_forest",  RandomForestRegressor(n_estimators=100, random_state=42)),
    ("gradient_boost", GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ("knn",            KNeighborsRegressor(n_neighbors=5))
]

print("Base Learners Defined ✅")
for name, model in base_learners:
    print(f"  • {name}")

In [ ]:
# ------------------------------------------------
# Define Meta Learner (Level 1 Model)
# ------------------------------------------------

meta_learner = Ridge(alpha=1.0)

print("Meta Learner Defined ✅")
print("  • Ridge Regression")

In [ ]:
# ------------------------------------------------
# Build Stacking Regressor
# ------------------------------------------------

stacking_model = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    passthrough=False
)

print("Stacking Regressor Built ✅")

In [ ]:
# ------------------------------------------------
# Train Stacking Model
# ------------------------------------------------

stacking_model.fit(X_train_scaled, y_train)

print("Stacking Model Trained ✅")

In [ ]:
# ------------------------------------------------
# Evaluate Stacking Model
# ------------------------------------------------

y_pred_stack = stacking_model.predict(X_test_scaled)

stack_r2   = r2_score(y_test, y_pred_stack)
stack_mae  = mean_absolute_error(y_test, y_pred_stack)
stack_rmse = np.sqrt(mean_squared_error(y_test, y_pred_stack))

print("Stacking Model Metrics")
print(f"  R² Score : {stack_r2:.4f}")
print(f"  MAE      : ${stack_mae:,.0f}")
print(f"  RMSE     : ${stack_rmse:,.0f}")

In [ ]:
# ------------------------------------------------
# Performance Comparison: Individual vs Stacking
# ------------------------------------------------

results = {}

for name, model in base_learners:
    model.fit(X_train_scaled, y_train)
    r2  = r2_score(y_test, model.predict(X_test_scaled))
    mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
    results[name] = {
        "R2":   round(r2 * 100, 2),
        "MAE":  round(mae, 0)
    }
    print(f"{name} → R²: {r2:.4f}  MAE: ${mae:,.0f}")

results["stacking"] = {
    "R2":  round(stack_r2 * 100, 2),
    "MAE": round(stack_mae, 0)
}
print(f"stacking  → R²: {stack_r2:.4f}  MAE: ${stack_mae:,.0f}")

In [ ]:
# ------------------------------------------------
# R² Score Comparison Chart
# ------------------------------------------------

model_names = list(results.keys())
r2_scores   = [v["R2"] for v in results.values()]

colors = ["#3f5efb","#3f5efb","#3f5efb","#3f5efb","#fc466b"]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, r2_scores, color=colors, edgecolor="white", linewidth=1.2)

for bar, val in zip(bars, r2_scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - 1,
        f"{val}%",
        ha='center', va='top',
        color='white', fontweight='bold', fontsize=11
    )

plt.title("R² Score Comparison", fontsize=16, fontweight='bold')
plt.xlabel("Model", fontsize=13)
plt.ylabel("R² Score (%)", fontsize=13)
plt.ylim(60, 105)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../static/comparison_chart.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart Saved ✅")

In [ ]:
# ------------------------------------------------
# Actual vs Predicted Plot
# ------------------------------------------------

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_stack, alpha=0.5, color='#3f5efb', edgecolors='white', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title("Actual vs Predicted Price", fontsize=14, fontweight='bold')
plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------
# Save Model
# ------------------------------------------------

pickle.dump(
    stacking_model,
    open("../models/Stacking_Regressor.pkl", "wb")
)

pickle.dump(
    scaler,
    open("../models/scaler.pkl", "wb")
)

pickle.dump(
    encoders,
    open("../models/label_encoders.pkl", "wb")
)

pickle.dump(
    results,
    open("../models/r2_results.pkl", "wb")
)

print("Model Saved Successfully ✅")